In [79]:
import sys
import math
import pandas as pd
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter as letter

sys.path.append('..')
from modules.source import PROCESSED_DATA_PATH, OUTPUT_FOLDER_PATH

analysis_dfs = pd.read_pickle(f"{PROCESSED_DATA_PATH}/analysis_dfs.pkl")

### Excel Output

In [80]:
def auto_fit_columns(ws, n_idx, min_row=4, padding=3):
   merged = {rc for rng in ws.merged_cells.ranges if rng.min_col != rng.max_col for rc in rng.cells}

   for col in ws.iter_cols(min_row=min_row):
      lengths = [len(f"{c.value:.4f}" if isinstance(c.value, (int, float)) else str(c.value))
                 for c in col if c.value is not None and (c.row, c.column) not in merged]
      width = max(lengths, default=10) + padding
      idx = col[0].column
      ws.column_dimensions[letter(idx)].width = max(width, 14) if idx <= n_idx else min(max(width, 10), 16)

def bands(index):
   s = pd.Series(index.get_level_values(0))
   return ((s != s.shift()).cumsum() % 2 == 1) & (index.nlevels > 1)

def add_banner(ws, row, text, font, span, height, wrap=False):
   cell = ws.cell(row, 1, text)
   cell.font = font
   cell.alignment = Alignment(None, "center", wrap_text=wrap)
   ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=span)
   ws.row_dimensions[row].height = height

with pd.ExcelWriter(f"{OUTPUT_FOLDER_PATH}/analysis_output.xlsx", "openpyxl") as writer:
   for sheet_name, content in analysis_dfs.items():
      df, title, caption = content["df"], content["title"], content["caption"]
      df.to_excel(writer, sheet_name=sheet_name, startrow=3)

      ws = writer.sheets[sheet_name]
      n_idx = df.index.nlevels
      n_cols = df.shape[1] + n_idx
      span = max(n_cols, 8)
      h = df.columns.nlevels
      first = h + 4 + (h > 1)     # first data row (MultiIndex columns add an index-names row)
      last = first + len(df) - 1  # last data row

      rows, cols = bands(df.index), bands(df.columns)

      for row in ws.iter_rows(4, last, 1, n_cols):
         for c in row:
            c.alignment = Alignment(c.alignment.horizontal, "center", wrap_text=c.alignment.wrap_text)
            i, j = c.row - first, c.column - n_idx - 1   # data-row / value-column position (negative in header / index)
            if (i >= 0 and rows[i]) ^ (j >= 0 and cols[j]):
               c.fill = PatternFill("solid", "F2F2F2")
            if i >= 0 and j >= 0 and isinstance(c.value, (int, float)):
               c.number_format = "0.0000"

      auto_fit_columns(ws, n_idx)
      for c in range(n_cols + 1, span + 1):
         ws.column_dimensions[letter(c)].width = 10

      add_banner(ws, 1, title, Font(sz=16, b=True, color="1F4E79"), span, 28)
      total = sum(ws.column_dimensions[letter(c)].width for c in range(1, span + 1))
      height = max(30, 18 * math.ceil(len(caption) / (total * 0.85)))
      add_banner(ws, 2, caption, Font(sz=10, i=True, color="595959"), span, height, True)